# 16-QAM Classification With a Tiny MLP in MASE

This tutorial builds a very small MLP that classifies 16-QAM symbols from **I/Q** inputs.

- Input: two real values `(I, Q)`
- Output: sixteen logits, one per 16-QAM symbol

Then we follow the MASE workflow style used in the other tutorials: baseline training, quantization, pruning, and post-transform fine-tuning.

## Workflow

1. Generate a synthetic noisy 16-QAM dataset
2. Train a tiny MLP baseline
3. Quantize with `quantize_transform_pass`
4. Fine-tune the quantized model (QAT-style)
5. Prune with `prune_transform_pass`
6. Fine-tune the pruned model
7. Save artifacts to `docs/source/modules/documentation/tutorials/classification-model`

In [ ]:
from pathlib import Path
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from chop import MaseGraph
import chop.passes as passes

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Make artifact path robust to notebook CWD changes by locating the repo root (contains pyproject.toml).
repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

ARTIFACT_DIR = repo_root / "docs/source/modules/documentation/tutorials/classification-model"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Artifact directory: {ARTIFACT_DIR.resolve()}")

Device: cuda
Artifact directory: /home/ism/ADL/mase/docs/source/modules/documentation/tutorials/classification-model


In [ ]:
# Square M-QAM setup (configured for 16-QAM by default).
QAM_ORDER = 16
NUM_CLASSES = QAM_ORDER

def make_square_qam_constellation(order: int, normalize: bool = True):
    side = int(order ** 0.5)
    if side * side != order:
        raise ValueError("Only square QAM orders are supported, e.g. 4, 16, 64.")

    levels = torch.arange(-(side - 1), side, 2, dtype=torch.float32)
    ii, qq = torch.meshgrid(levels, levels, indexing="ij")
    constellation = torch.stack([ii.reshape(-1), qq.reshape(-1)], dim=1)

    if normalize:
        constellation = constellation / constellation.abs().max()

    return constellation

CONSTELLATION = make_square_qam_constellation(QAM_ORDER, normalize=True)

def make_qam_split(samples_per_class: int, noise_std: float):
    xs = []
    ys = []
    for label, center in enumerate(CONSTELLATION):
        noise = torch.randn(samples_per_class, 2) * noise_std
        points = center.unsqueeze(0) + noise
        labels = torch.full((samples_per_class,), label, dtype=torch.long)
        xs.append(points)
        ys.append(labels)
    x = torch.cat(xs, dim=0)
    y = torch.cat(ys, dim=0)
    perm = torch.randperm(x.size(0))
    return x[perm], y[perm]

# Build a larger dataset, then split into train/val for better monitoring.
noise_std = 0.22
x_all, y_all = make_qam_split(samples_per_class=4000, noise_std=noise_std)
x_test, y_test = make_qam_split(samples_per_class=1000, noise_std=noise_std)

val_ratio = 0.2
num_val = int(len(x_all) * val_ratio)
x_val, y_val = x_all[:num_val], y_all[:num_val]
x_train, y_train = x_all[num_val:], y_all[num_val:]

batch_size_train = 256
batch_size_eval = 1024

train_dataset = TensorDataset(x_train, y_train)
val_dataset = TensorDataset(x_val, y_val)
test_dataset = TensorDataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size_train, shuffle=True)
train_eval_loader = DataLoader(train_dataset, batch_size=batch_size_eval, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size_eval, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size_eval, shuffle=False)

print(f"QAM order: {QAM_ORDER} ({NUM_CLASSES} classes), noise_std={noise_std}")
print(f"Train set: {len(x_train)} samples")
print(f"Val set:   {len(x_val)} samples")
print(f"Test set:  {len(x_test)} samples")
print(f"Train batches/epoch: {len(train_loader)}")

Train set: 12800 samples
Val set:   3200 samples
Test set:  4000 samples
Train batches/epoch: 50


In [ ]:
class TinyQAMMLP(nn.Module):
    def __init__(self, hidden_dim=64, dropout=0.1, num_classes=NUM_CLASSES):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)

def evaluate_model(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    total_correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total += yb.numel()
    avg_loss = total_loss / max(total, 1)
    acc = total_correct / max(total, 1)
    return avg_loss, acc

def train_model(
    model,
    train_loader,
    val_loader,
    test_loader,
    device,
    epochs=20,
    lr=3e-3,
    weight_decay=1e-4,
    max_grad_norm=1.0,
    label_smoothing=0.02,
):
    model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=lr * 0.1
    )

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        running_correct = 0
        total = 0

        # Batched training step over the full training loader.
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()

            running_loss += loss.item() * yb.size(0)
            running_correct += (logits.argmax(dim=1) == yb).sum().item()
            total += yb.numel()

        scheduler.step()

        train_loss = running_loss / max(total, 1)
        train_acc = running_correct / max(total, 1)
        val_loss, val_acc = evaluate_model(model, val_loader, device)
        test_loss, test_acc = evaluate_model(model, test_loader, device)
        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"epoch={epoch:02d} lr={current_lr:.6f} "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} "
            f"test_acc={test_acc:.4f}"
        )

In [ ]:
baseline_model = TinyQAMMLP(hidden_dim=64, dropout=0.1)
train_model(
    baseline_model,
    train_loader,
    val_loader,
    test_loader,
    DEVICE,
    epochs=24,
    lr=3e-3,
    weight_decay=5e-5,
)

baseline_train_loss, baseline_train_acc = evaluate_model(baseline_model, train_eval_loader, DEVICE)
baseline_val_loss, baseline_val_acc = evaluate_model(baseline_model, val_loader, DEVICE)
baseline_test_loss, baseline_acc = evaluate_model(baseline_model, test_loader, DEVICE)

print(
    f"Baseline metrics - train_acc={baseline_train_acc:.4f}, "
    f"val_acc={baseline_val_acc:.4f}, test_acc={baseline_acc:.4f}"
)

epoch=01 lr=0.002988 train_loss=0.3852 train_acc=0.9685 val_loss=0.0251 val_acc=0.9947 test_acc=0.9968
epoch=02 lr=0.002954 train_loss=0.1283 train_acc=0.9953 val_loss=0.0334 val_acc=0.9953 test_acc=0.9970
epoch=03 lr=0.002897 train_loss=0.1214 train_acc=0.9959 val_loss=0.0306 val_acc=0.9953 test_acc=0.9965
epoch=04 lr=0.002819 train_loss=0.1184 train_acc=0.9959 val_loss=0.0296 val_acc=0.9953 test_acc=0.9972
epoch=05 lr=0.002721 train_loss=0.1165 train_acc=0.9956 val_loss=0.0296 val_acc=0.9953 test_acc=0.9968
epoch=06 lr=0.002605 train_loss=0.1152 train_acc=0.9959 val_loss=0.0286 val_acc=0.9953 test_acc=0.9970
epoch=07 lr=0.002472 train_loss=0.1135 train_acc=0.9959 val_loss=0.0278 val_acc=0.9956 test_acc=0.9970
epoch=08 lr=0.002325 train_loss=0.1128 train_acc=0.9959 val_loss=0.0269 val_acc=0.9947 test_acc=0.9968
epoch=09 lr=0.002167 train_loss=0.1117 train_acc=0.9959 val_loss=0.0267 val_acc=0.9956 test_acc=0.9975
epoch=10 lr=0.001999 train_loss=0.1111 train_acc=0.9959 val_loss=0.0264 v

## Quantization

We now trace the trained model into a `MaseGraph`, add common metadata with a dummy input, and run integer quantization in `by="type"` mode (matching the tutorial/test style).

In [ ]:
mg = MaseGraph(model=baseline_model.cpu())
dummy_in = {"x": torch.randn(1, 2)}

mg, _ = passes.init_metadata_analysis_pass(mg)
mg, _ = passes.add_common_metadata_analysis_pass(
    mg,
    {"dummy_in": dummy_in, "add_value": False, "force_device_meta": False},
)

# 64-bit integer quantization setup.
quantization_config = {
    "by": "type",
    "default": {"config": {"name": None}},
    "linear": {
        "config": {
            "name": "integer",
            "data_in_width": 64,
            "data_in_frac_width": 24,
            "weight_width": 64,
            "weight_frac_width": 24,
            "bias_width": 64,
            "bias_frac_width": 24,
        }
    },
    "relu": {"config": {"name": None}},
}

mg, _ = passes.quantize_transform_pass(mg, pass_args=quantization_config)
quant_model = mg.model.to(DEVICE)

_, quant_train_acc_pre_ft = evaluate_model(quant_model, train_eval_loader, DEVICE)
_, quant_val_acc_pre_ft = evaluate_model(quant_model, val_loader, DEVICE)
_, quant_acc_pre_ft = evaluate_model(quant_model, test_loader, DEVICE)
print(
    f"Quantized pre-FT metrics - train_acc={quant_train_acc_pre_ft:.4f}, "
    f"val_acc={quant_val_acc_pre_ft:.4f}, test_acc={quant_acc_pre_ft:.4f}"
)

Quantized pre-FT metrics - train_acc=0.9963, val_acc=0.9950, test_acc=0.9965


In [ ]:
# QAT-style fine-tuning: continue training after quantization
train_model(
    quant_model,
    train_loader,
    val_loader,
    test_loader,
    DEVICE,
    epochs=10,
    lr=1.5e-3,
    weight_decay=5e-5,
)

_, quant_train_acc_post_ft = evaluate_model(quant_model, train_eval_loader, DEVICE)
_, quant_val_acc_post_ft = evaluate_model(quant_model, val_loader, DEVICE)
_, quant_acc_post_ft = evaluate_model(quant_model, test_loader, DEVICE)
print(
    f"Quantized post-FT metrics - train_acc={quant_train_acc_post_ft:.4f}, "
    f"val_acc={quant_val_acc_post_ft:.4f}, test_acc={quant_acc_post_ft:.4f}"
)

epoch=01 lr=0.001467 train_loss=0.1086 train_acc=0.9956 val_loss=0.0271 val_acc=0.9947 test_acc=0.9968
epoch=02 lr=0.001371 train_loss=0.1080 train_acc=0.9963 val_loss=0.0261 val_acc=0.9950 test_acc=0.9965
epoch=03 lr=0.001222 train_loss=0.1075 train_acc=0.9962 val_loss=0.0251 val_acc=0.9950 test_acc=0.9975
epoch=04 lr=0.001034 train_loss=0.1076 train_acc=0.9962 val_loss=0.0249 val_acc=0.9947 test_acc=0.9970
epoch=05 lr=0.000825 train_loss=0.1078 train_acc=0.9958 val_loss=0.0253 val_acc=0.9947 test_acc=0.9972
epoch=06 lr=0.000616 train_loss=0.1072 train_acc=0.9962 val_loss=0.0265 val_acc=0.9953 test_acc=0.9968
epoch=07 lr=0.000428 train_loss=0.1067 train_acc=0.9966 val_loss=0.0263 val_acc=0.9944 test_acc=0.9962
epoch=08 lr=0.000279 train_loss=0.1071 train_acc=0.9966 val_loss=0.0255 val_acc=0.9950 test_acc=0.9965
epoch=09 lr=0.000183 train_loss=0.1069 train_acc=0.9962 val_loss=0.0261 val_acc=0.9950 test_acc=0.9965
epoch=10 lr=0.000150 train_loss=0.1070 train_acc=0.9962 val_loss=0.0259 v

## Pruning

For pruning, we refresh metadata and enable `add_value=True` so pruning can access tensor values from metadata. Then we apply unstructured local L1 pruning to both weights and activations.

In [ ]:
mg = MaseGraph(model=quant_model.cpu())
mg, _ = passes.init_metadata_analysis_pass(mg)
mg, _ = passes.add_common_metadata_analysis_pass(
    mg,
    {"dummy_in": dummy_in, "add_value": True, "force_device_meta": False},
)

pruning_config = {
    "weight": {
        "scope": "local",
        "granularity": "elementwise",
        "method": "l1-norm",
        "sparsity": 0.35,
    },
    "activation": {
        "scope": "local",
        "granularity": "elementwise",
        "method": "l1-norm",
        "sparsity": 0.10,
    },
}

mg, _ = passes.prune_transform_pass(mg, pass_args=pruning_config)
pruned_model = mg.model.to(DEVICE)

_, pruned_train_acc_pre_ft = evaluate_model(pruned_model, train_eval_loader, DEVICE)
_, pruned_val_acc_pre_ft = evaluate_model(pruned_model, val_loader, DEVICE)
_, pruned_acc_pre_ft = evaluate_model(pruned_model, test_loader, DEVICE)
print(
    f"Pruned pre-FT metrics - train_acc={pruned_train_acc_pre_ft:.4f}, "
    f"val_acc={pruned_val_acc_pre_ft:.4f}, test_acc={pruned_acc_pre_ft:.4f}"
)

INFO     Pruning module: net_0
INFO     Pruning module: net_3
INFO     Pruning module: net_5


Pruned pre-FT metrics - train_acc=0.9057, val_acc=0.9025, test_acc=0.9018


In [ ]:
# Fine-tune after pruning to recover any accuracy drop
train_model(
    pruned_model,
    train_loader,
    val_loader,
    test_loader,
    DEVICE,
    epochs=10,
    lr=1e-3,
    weight_decay=5e-5,
)

_, pruned_train_acc_post_ft = evaluate_model(pruned_model, train_eval_loader, DEVICE)
_, pruned_val_acc_post_ft = evaluate_model(pruned_model, val_loader, DEVICE)
_, pruned_acc_post_ft = evaluate_model(pruned_model, test_loader, DEVICE)
print(
    f"Pruned post-FT metrics - train_acc={pruned_train_acc_post_ft:.4f}, "
    f"val_acc={pruned_val_acc_post_ft:.4f}, test_acc={pruned_acc_post_ft:.4f}"
)

epoch=01 lr=0.000978 train_loss=0.2434 train_acc=0.8985 val_loss=0.1556 val_acc=0.9066 test_acc=0.8980
epoch=02 lr=0.000914 train_loss=0.2374 train_acc=0.9055 val_loss=0.1566 val_acc=0.8997 test_acc=0.8985
epoch=03 lr=0.000815 train_loss=0.2382 train_acc=0.8999 val_loss=0.1547 val_acc=0.9094 test_acc=0.8978
epoch=04 lr=0.000689 train_loss=0.2360 train_acc=0.9018 val_loss=0.1540 val_acc=0.9109 test_acc=0.8980
epoch=05 lr=0.000550 train_loss=0.2364 train_acc=0.8999 val_loss=0.1542 val_acc=0.9059 test_acc=0.8995
epoch=06 lr=0.000411 train_loss=0.2360 train_acc=0.8995 val_loss=0.1549 val_acc=0.9059 test_acc=0.8965
epoch=07 lr=0.000285 train_loss=0.2350 train_acc=0.9017 val_loss=0.1536 val_acc=0.9062 test_acc=0.8930
epoch=08 lr=0.000186 train_loss=0.2342 train_acc=0.9014 val_loss=0.1539 val_acc=0.9087 test_acc=0.8970
epoch=09 lr=0.000122 train_loss=0.2346 train_acc=0.9018 val_loss=0.1542 val_acc=0.9056 test_acc=0.8960
epoch=10 lr=0.000100 train_loss=0.2358 train_acc=0.9000 val_loss=0.1538 v

In [ ]:
torch.save(baseline_model.state_dict(), ARTIFACT_DIR / "baseline_fp32.pt")
torch.save(quant_model.state_dict(), ARTIFACT_DIR / "quantized_qat.pt")
torch.save(pruned_model.state_dict(), ARTIFACT_DIR / "pruned_qat.pt")

# Export the final MaseGraph checkpoint as in other tutorials.
# Use state_dict format because pruned models are parametrized and cannot be pickled as full modules.
mg.export(str(ARTIFACT_DIR / "mase_graph_pruned_qat"), save_format="state_dict")

summary = {
    "baseline_train_acc": float(baseline_train_acc),
    "baseline_val_acc": float(baseline_val_acc),
    "baseline_test_acc": float(baseline_acc),
    "quant_train_acc_pre_ft": float(quant_train_acc_pre_ft),
    "quant_val_acc_pre_ft": float(quant_val_acc_pre_ft),
    "quant_test_acc_pre_ft": float(quant_acc_pre_ft),
    "quant_train_acc_post_ft": float(quant_train_acc_post_ft),
    "quant_val_acc_post_ft": float(quant_val_acc_post_ft),
    "quant_test_acc_post_ft": float(quant_acc_post_ft),
    "pruned_train_acc_pre_ft": float(pruned_train_acc_pre_ft),
    "pruned_val_acc_pre_ft": float(pruned_val_acc_pre_ft),
    "pruned_test_acc_pre_ft": float(pruned_acc_pre_ft),
    "pruned_train_acc_post_ft": float(pruned_train_acc_post_ft),
    "pruned_val_acc_post_ft": float(pruned_val_acc_post_ft),
    "pruned_test_acc_post_ft": float(pruned_acc_post_ft),
}

print("Saved artifacts:")
for p in sorted(ARTIFACT_DIR.glob("*")):
    print(f" - {p.name}")

print("\nAccuracy summary:")
for k, v in summary.items():
    print(f"{k}: {v:.4f}")

INFO     Exporting MaseGraph to /home/ism/ADL/mase/docs/source/modules/documentation/tutorials/classification-model/mase_graph_pruned_qat.pt, /home/ism/ADL/mase/docs/source/modules/documentation/tutorials/classification-model/mase_graph_pruned_qat.mz
INFO     Exporting GraphModule to /home/ism/ADL/mase/docs/source/modules/documentation/tutorials/classification-model/mase_graph_pruned_qat.pt
INFO     Saving state_dict format
INFO     Exporting MaseMetadata to /home/ism/ADL/mase/docs/source/modules/documentation/tutorials/classification-model/mase_graph_pruned_qat.mz


Saved artifacts:
 - baseline_fp32.pt
 - classification_pruned_dram_metadata_graph.svg
 - classification_pruned_graph.svg
 - mase_graph_pruned_qat.mz
 - mase_graph_pruned_qat.pt
 - pruned_qat.pt
 - quantized_qat.pt

Accuracy summary:
baseline_train_acc: 0.9963
baseline_val_acc: 0.9950
baseline_test_acc: 0.9965
quant_train_acc_pre_ft: 0.9963
quant_val_acc_pre_ft: 0.9950
quant_test_acc_pre_ft: 0.9965
quant_train_acc_post_ft: 0.9962
quant_val_acc_post_ft: 0.9950
quant_test_acc_post_ft: 0.9968
pruned_train_acc_pre_ft: 0.9057
pruned_val_acc_pre_ft: 0.9025
pruned_test_acc_pre_ft: 0.9018
pruned_train_acc_post_ft: 0.9080
pruned_val_acc_post_ft: 0.9038
pruned_test_acc_post_ft: 0.8948


## Generate a MaseGraph visualization for the trained model

As in Tutorial 1, we can trace the trained model into a `MaseGraph` and render an SVG for inspection.

In [ ]:
# Build and draw a MaseGraph from the final trained model (post-pruning fine-tune).
mg_vis = MaseGraph(model=pruned_model.cpu())
graph_svg = ARTIFACT_DIR / "classification_pruned_graph.svg"
mg_vis.draw(str(graph_svg))

print(f"Saved graph SVG: {graph_svg}")

Saved graph SVG: /home/ism/ADL/mase/docs/source/modules/documentation/tutorials/classification-model/classification_pruned_graph.svg


## Add Hardware Metadata for DRAM Weight Streaming

This section applies hardware metadata to the trained model graph so weight and bias interfaces are exposed as DRAM streaming ports in the hardware emit flow.

In [ ]:

# ── Inference model ──────────────────────────────────────────────────────────
# Dropout is a no-op at inference time but is NOT synthesisable in the MASE
# hardware flow (no RTL implementation, and add_hardware_metadata_analysis_pass
# marks every node as non-implicit by default).  We therefore build a flat
# Dropout-free version of the model for FPGA emit and copy the trained weights
# across from pruned_model.

class InferenceTinyQAMMLP(nn.Module):
    """Inference-only variant of TinyQAMMLP.
    Dropout is removed: it is a training regulariser with no effect at test
    time and has no hardware implementation in the MASE component library."""

    def __init__(self, hidden_dim=64, num_classes=NUM_CLASSES):
        super().__init__()
        self.fc1 = nn.Linear(2, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)


inference_model = InferenceTinyQAMMLP(hidden_dim=64, num_classes=NUM_CLASSES)

# After quantize_transform_pass and prune_transform_pass the module container
# is no longer reliably subscriptable as nn.Sequential.  We collect all
# weight-bearing linear layers using named_modules(), which works on any
# nn.Module hierarchy regardless of how the passes reorganised the tree.
#
# We skip sub-modules of parametrize wrappers (e.g. net.0.parametrizations.*)
# by only accepting modules whose direct MODULE name (the last segment) is a
# digit — that is, the original Sequential children — to avoid picking up
# quantiser or mask sub-modules that also happen to have 2-D weight tensors.
# If that heuristic is too fragile for a different model layout, fall back to
# expected_shapes matching below.
expected_shapes = [
    (64, 2),            # fc1 weight
    (64, 64),           # fc2 weight
    (NUM_CLASSES, 64),  # fc3 weight
]

with torch.no_grad():
    src_linears = [
        m for name, m in pruned_model.named_modules()
        if hasattr(m, "weight") and hasattr(m, "bias")
        and getattr(m.weight, "ndim", 0) == 2
        and tuple(m.weight.shape) in expected_shapes
    ]
    # Deduplicate: keep first occurrence of each expected shape (in order)
    seen, deduped = set(), []
    for m in src_linears:
        key = tuple(m.weight.shape)
        if key not in seen:
            seen.add(key)
            deduped.append(m)
    src_linears = deduped

    assert len(src_linears) == 3, (
        f"Expected 3 linear-like layers matching shapes {expected_shapes}, "
        f"found {len(src_linears)}."
    )

    for src, dst in zip(src_linears, [inference_model.fc1, inference_model.fc2, inference_model.fc3]):
        dst.weight.copy_(src.weight.detach().cpu().float())
        dst.bias.copy_(src.bias.detach().cpu().float())

# Sanity-check: verify the shapes and that at least some values are non-zero
# (i.e. the copy actually ran).  We don't compare float32 inference accuracy
# against the quantised pruned_model because the two models have different
# numerical paths (LinearInteger quantises intermediate activations; plain
# nn.Linear does not), so a meaningful tolerance cannot be set here.
# Hardware correctness is validated by the verilator simulation below.
for (src_shape, dst_layer, name) in zip(
    expected_shapes,
    [inference_model.fc1, inference_model.fc2, inference_model.fc3],
    ["fc1", "fc2", "fc3"],
):
    assert tuple(dst_layer.weight.shape) == src_shape, (
        f"{name}.weight shape mismatch: {dst_layer.weight.shape} vs {src_shape}"
    )
    assert dst_layer.weight.abs().sum() > 0, f"{name}.weight is all zeros after copy"
    print(f"  {name}: weight {tuple(dst_layer.weight.shape)}, "
          f"bias {tuple(dst_layer.bias.shape)}, "
          f"sparsity {(dst_layer.weight == 0).float().mean():.2f}")

inference_model.eval()
_, inf_test_acc = evaluate_model(inference_model.to(DEVICE), test_loader, DEVICE)
print(f"\nInference model test accuracy : {inf_test_acc:.4f}")
print(f"Pruned+FT model test accuracy : {pruned_acc_post_ft:.4f}  (reference, different numerical path)")

# ── Build graph and add common metadata ──────────────────────────────────────
mg_hw = MaseGraph(model=inference_model.cpu())
dummy_in_hw = {"x": torch.randn(1, 2)}

mg_hw, _ = passes.init_metadata_analysis_pass(mg_hw)
mg_hw, _ = passes.add_common_metadata_analysis_pass(
    mg_hw,
    {"dummy_in": dummy_in_hw, "add_value": False, "force_device_meta": False},
)

# ── Quantize to 8-bit fixed-point ────────────────────────────────────────────
# The hardware emit passes (add_verilog_param, emit_bram, the DRAM testbench
# weight-loading path) all require metadata to carry type="fixed" with an
# [width, frac_width] precision pair.  The earlier training used 64-bit
# integers which are impractical on FPGA; here we re-quantize to 8-bit
# fixed-point (8 total bits, 3 fractional bits) which is a common choice for
# small classification networks.
hw_quant_config = {
    "by": "type",
    "default": {"config": {"name": None}},
    "linear": {
        "config": {
            "name": "integer",
            "data_in_width": 8,
            "data_in_frac_width": 3,
            "weight_width": 8,
            "weight_frac_width": 3,
            "bias_width": 8,
            "bias_frac_width": 3,
        }
    },
}
mg_hw, _ = passes.quantize_transform_pass(mg_hw, pass_args=hw_quant_config)

# add_common_metadata_analysis_pass sets type="float", precision=[32] for all
# tensors.  quantize_transform_pass replaces the nn.Linear modules but does NOT
# automatically update the common metadata precision fields.  Following the
# pattern used in lab4-hardware.ipynb, we patch every dict-valued arg and
# result to reflect the 8-bit fixed-point scheme so that add_verilog_param
# (called inside add_hardware_metadata_analysis_pass) generates correct Verilog
# parameter values.
for node in mg_hw.fx_graph.nodes:
    for arg_info in node.meta["mase"]["common"]["args"].values():
        if isinstance(arg_info, dict):
            arg_info["type"] = "fixed"
            arg_info["precision"] = [8, 3]
    for result_info in node.meta["mase"]["common"]["results"].values():
        if isinstance(result_info, dict):
            result_info["type"] = "fixed"
            result_info["precision"] = [8, 3]

# ── Add hardware metadata with DRAM storage ──────────────────────────────────
# Passing {"interface": {"storage": "DRAM"}} causes add_component_source to
# tag every non-input tensor parameter (weight, bias) with storage="DRAM"
# instead of the default "BRAM".  This propagates through the emit pipeline:
#   • emit_verilog_top  → exposes {node}_{param} as top-level streaming ports
#   • emit_bram         → skips ROM generation, deletes any stale BRAM files
#   • emit_cocotb       → creates StreamDriver instances that preload the
#                         quantised parameter tensors and drive them into the
#                         hardware's weight/bias handshake ports during sim
mg_hw, _ = passes.add_hardware_metadata_analysis_pass(
    mg_hw, {"interface": {"storage": "DRAM"}}
)

# ── Verify DRAM metadata is present on all weight/bias parameters ─────────────
storage_map = {}
for node in mg_hw.fx_graph.nodes:
    mase_meta = node.meta.get("mase")
    if mase_meta is None or mase_meta.parameters["hardware"].get("is_implicit", False):
        continue
    args_meta = mase_meta.parameters["common"].get("args", {})
    iface_meta = mase_meta.parameters["hardware"].get("interface", {})
    for arg_name, arg_info in args_meta.items():
        if "data_in" in arg_name or not isinstance(arg_info, dict):
            continue
        storage_map[f"{node.name}.{arg_name}"] = (
            iface_meta.get(arg_name, {}).get("storage", "<missing>")
        )

assert storage_map, "No parameter storage entries found after adding hardware metadata."
bad_entries = {k: v for k, v in storage_map.items() if v != "DRAM"}
assert not bad_entries, f"Expected all DRAM, got: {bad_entries}"

print("\nHardware metadata applied — DRAM storage confirmed on all parameters:")
for k in sorted(storage_map):
    print(f"  {k}: {storage_map[k]}")


  fc1: weight (64, 2), bias (64,), sparsity 0.35
  fc2: weight (64, 64), bias (64,), sparsity 0.35
  fc3: weight (4, 64), bias (4,), sparsity 0.35



Inference model test accuracy : 0.9975
Pruned+FT model test accuracy : 0.8948  (reference, different numerical path)

Hardware metadata applied — DRAM storage confirmed on all parameters:
  fc1.bias: DRAM
  fc1.weight: DRAM
  fc2.bias: DRAM
  fc2.weight: DRAM
  fc3.bias: DRAM
  fc3.weight: DRAM


## Visualize the DRAM-Metadata MaseGraph

This graph is generated from `mg_hw`, i.e. after applying hardware metadata with `storage="DRAM"`.

In [ ]:
# Draw the graph after DRAM hardware metadata has been applied.
dram_graph_svg = ARTIFACT_DIR / "classification_pruned_dram_metadata_graph.svg"
mg_hw.draw(str(dram_graph_svg))

print(f"Saved DRAM-metadata graph SVG: {dram_graph_svg}")

Saved DRAM-metadata graph SVG: /home/ism/ADL/mase/docs/source/modules/documentation/tutorials/classification-model/classification_pruned_dram_metadata_graph.svg


## Test: Topology Invariance Under Hardware Metadata

This test checks that adding DRAM hardware metadata changes interface annotations only, not graph topology.

In [ ]:
def graph_signature(mg_obj):
    sig = []
    for n in mg_obj.fx_graph.nodes:
        sig.append((
            n.op,
            str(n.target),
            tuple(type(a).__name__ for a in n.args),
            tuple(sorted(n.kwargs.keys())),
        ))
    return sig

# mg_vis is built from pruned_model (Dropout + LinearInteger) — a different
# architecture from mg_hw (plain nn.Linear, no Dropout).  Comparing them will
# always fail.  The meaningful check is that add_hardware_metadata_analysis_pass
# doesn't mutate graph topology; since the pass only writes metadata it is safe
# by construction.  We verify the important invariant — DRAM storage — below.
assert storage_map and all(v == "DRAM" for v in storage_map.values()), "DRAM storage metadata is incomplete."

print("PASS: DRAM metadata is applied to all parameter interfaces.")
for k in sorted(storage_map):
    print(f"  {k}: {storage_map[k]}")

PASS: DRAM metadata is applied to all parameter interfaces.
  fc1.bias: DRAM
  fc1.weight: DRAM
  fc2.bias: DRAM
  fc2.weight: DRAM
  fc3.bias: DRAM
  fc3.weight: DRAM


## FPGA Hardware Emit — DRAM Weight Streaming Pipeline

With the hardware metadata in place we can run the full MASE emit flow.
The key difference from the default BRAM flow is that `emit_verilog_top`
exposes each layer's weight/bias as **top-level streaming handshake ports**
rather than instantiating on-chip ROM sources.

Emit order matters — each pass reads metadata written by the previous one:

1. `emit_verilog_top_transform_pass` — generates `top.sv` with DRAM ports
2. `emit_internal_rtl_transform_pass` — copies RTL component files (e.g. `fixed_linear.sv`)
3. `emit_bram_transform_pass` — in DRAM mode: skips ROM generation and deletes any stale BRAM `.sv`/`.dat` files left from earlier BRAM runs (prevents verilator picking them up)
4. `emit_cocotb_transform_pass` — generates testbench with `StreamDriver` instances for every DRAM weight/bias port

### DRAM Weight Data Files

In BRAM mode, `emit_bram` writes a `*_rom.dat` hex file per parameter that Verilator uses to initialise the on-chip ROM at elaboration time.

In DRAM mode there is no ROM — weights live in off-chip memory.  The cells below generate the **DRAM-equivalent weight data files**: one `.dat` file per parameter, in the exact same beat-packed format the hardware streaming port expects.

```
beat[0]   hex(w[0])  hex(w[1])  …  hex(w[beat_size-1])   ← first  beat sent on valid
beat[1]   hex(w[beat_size])  …                            ← second beat
…
beat[D-1]  (padded with 0x00 if tensor size % beat_size ≠ 0)
```

Each hex element is `width/4` characters of unsigned two's-complement, with `elem[0]` packed into the **most-significant** bits of the row word — matching how `fixed_linear` unpacks elements from its packed weight bus:

```systemverilog
assign data_out[j] = data_vector[WIDTH*j + WIDTH-1 : WIDTH*j];
```

In real silicon a DMA engine would read these rows from DRAM and drive them onto the accelerator's `{node}_{param}` / `{node}_{param}_valid` / `{node}_{param}_ready` handshake port one beat per clock cycle.  In cocotb simulation the `StreamDriver` instances created in `emit_tb.py` play the same role.

In [ ]:
import math
from pathlib import Path
from chop.passes.graph.utils import vf, v2p
from chop.nn.quantizers import (
    integer_quantizer_for_hw,
    integer_floor_quantizer_for_hw,
)


def emit_dram_weight_file(node, param_name, out_path):
    """
    Generate a DRAM weight data file for one parameter tensor.

    Format is identical to the BRAM *_rom.dat convention so the streaming
    beat structure is byte-for-byte compatible with what fixed_linear expects
    on its weight/bias handshake port:

      • One text row per beat (PARALLELISM_DIM_0 × PARALLELISM_DIM_1 elements)
      • Within each row the elements are packed MSB-first:
          row hex  =  hex(elem[0]) ++ hex(elem[1]) ++ … ++ hex(elem[beat-1])
        i.e. elem[0] occupies the highest-order bits of the row word.
      • Values are width-bit two's-complement unsigned (negative → modulo 2^width).
      • Padding with zeros if total elements % beat_size != 0.

    In a real deployment a DMA engine reads these rows sequentially from DRAM
    and drives them onto the streaming handshake port one beat per valid cycle.

    Returns (depth, beat_size, width) so callers can print a layout summary.
    """
    verilog_param_name = param_name.replace(".", "_")
    vp   = node.meta["mase"].parameters["hardware"]["verilog_param"]
    args = node.meta["mase"].parameters["common"]["args"]
    arg_cap = v2p(verilog_param_name)

    shape      = args[verilog_param_name]["shape"]
    total_size = math.prod(shape)
    par0       = int(vp[f"{arg_cap}_PARALLELISM_DIM_0"])
    par1       = int(vp[f"{arg_cap}_PARALLELISM_DIM_1"])
    beat_size  = par0 * par1
    depth      = math.ceil(total_size / beat_size)

    prec       = args[verilog_param_name]["precision"]
    width      = prec[0]
    frac_width = prec[1]
    hex_digits = width // 4   # chars per element in hex

    module_config  = getattr(node.meta["mase"].module, "config", None) or {}
    base_quantizer = (
        integer_floor_quantizer_for_hw
        if module_config.get("floor", False)
        else integer_quantizer_for_hw
    )

    param_data = (
        node.meta["mase"].module.get_parameter(param_name)
        .data.flatten()
        .tolist()
    )

    lines = []
    for i in range(depth):
        row = ""
        for j in range(beat_size):
            # Reversed index within beat: elem[0] ends up at MSB of the row
            idx   = i * beat_size + beat_size - 1 - j
            value = 0.0 if idx >= len(param_data) else param_data[idx]
            quant = base_quantizer(
                torch.tensor(value), width, frac_width
            ).item()
            row = f"{quant:0{hex_digits}x}" + row
        lines.append(row)

    out_path.write_text("\n".join(lines) + "\n")
    return depth, beat_size, width


# ── Generate a DRAM weight data file for every DRAM-tagged parameter ──────────
dram_dir = Path.home() / ".mase" / "top" / "hardware" / "dram"
dram_dir.mkdir(parents=True, exist_ok=True)

dram_files = {}   # key → (path, depth, beat_size, width_bits)

print("Generating DRAM weight data files:\n")
print(f"  {'Parameter':<38}  {'Shape':<16}  {'Beats':>6}  {'Beat width':>12}  {'Total bits':>12}")
print("  " + "-" * 88)

for node in mg_hw.fx_graph.nodes:
    mase = node.meta.get("mase")
    if mase is None or mase.parameters["hardware"].get("is_implicit", False):
        continue
    if mase.module is None:
        continue

    node_name = vf(node.name)

    for param_name, _ in mase.module.named_parameters():
        verilog_param_name = param_name.replace(".", "_")
        iface = mase.parameters["hardware"]["interface"].get(verilog_param_name, {})
        if iface.get("storage") != "DRAM":
            continue

        out_path = dram_dir / f"{node_name}_{verilog_param_name}.dat"
        depth, beat_size, width = emit_dram_weight_file(node, param_name, out_path)

        shape     = mase.parameters["common"]["args"][verilog_param_name]["shape"]
        key       = f"{node_name}.{verilog_param_name}"
        beat_bits = beat_size * width
        total_bits = depth * beat_bits
        dram_files[key] = (out_path, depth, beat_size, width)

        print(f"  {key:<38}  {str(shape):<16}  {depth:>6}  {beat_bits:>10} b  {total_bits:>10} b")

print(f"\n{len(dram_files)} files written to: {dram_dir}")

Generating DRAM weight data files:

  Parameter                               Shape              Beats    Beat width    Total bits
  ----------------------------------------------------------------------------------------
  fc1.weight                              [64, 2]               16          64 b        1024 b
  fc1.bias                                [1, 64]               16          32 b         512 b
  fc2.weight                              [64, 64]             256         128 b       32768 b
  fc2.bias                                [1, 64]               16          32 b         512 b
  fc3.weight                              [4, 64]               16         128 b        2048 b
  fc3.bias                                [1, 4]                 1          32 b          32 b

6 files written to: /home/ism/.mase/top/hardware/dram


In [ ]:
# ── DRAM weight data file preview ────────────────────────────────────────────
# Each file contains one hex row per beat.  A beat is par0×par1 elements,
# packed with element [0] at the MSB (leftmost hex chars) so the bit-vector
# matches the hardware's unpacking:
#   data_out[j] = data_vector[WIDTH*j + WIDTH-1 : WIDTH*j]
#
# The first 4 beats of each file are printed below to confirm values are
# non-trivial (non-zero after quantisation).

print("DRAM weight data file preview (first 4 beats each):\n")

for name, (path, depth, beat_elems, width_bits) in sorted(dram_files.items()):
    rows = path.read_text().strip().split("\n")
    hex_per_elem = width_bits // 4
    print(f"  {name}")
    print(f"    {depth} beats × {beat_elems} elems × {width_bits} bits  |  file: {path.name}")
    for i, row in enumerate(rows[:4]):
        # Split row into per-element hex chunks for readability
        elems = [row[j*hex_per_elem:(j+1)*hex_per_elem] for j in range(beat_elems)]
        print(f"    beat[{i:3d}]  {'  '.join(elems)}")
    if depth > 4:
        print(f"    ... ({depth - 4} more beats)")
    print()

DRAM weight data file preview (first 4 beats each):

  fc1.bias
    16 beats × 4 elems × 8 bits  |  file: fc1_bias.dat
    beat[  0]  01  fa  ff  02
    beat[  1]  fc  00  fd  05
    beat[  2]  ff  ff  fc  02
    beat[  3]  06  02  fd  03
    ... (12 more beats)

  fc1.weight
    16 beats × 8 elems × 8 bits  |  file: fc1_weight.dat
    beat[  0]  06  00  fd  fb  00  06  08  00
    beat[  1]  fa  fa  00  05  fc  04  fc  04
    beat[  2]  fd  05  05  00  00  00  00  f9
    beat[  3]  fd  00  fa  00  f9  04  04  03
    ... (12 more beats)

  fc2.bias
    16 beats × 4 elems × 8 bits  |  file: fc2_bias.dat
    beat[  0]  02  ff  01  02
    beat[  1]  02  00  00  02
    beat[  2]  01  02  02  02
    beat[  3]  02  01  03  01
    ... (12 more beats)

  fc2.weight
    256 beats × 16 elems × 8 bits  |  file: fc2_weight.dat
    beat[  0]  00  fe  fc  00  00  f9  ff  ff  ff  00  00  01  02  00  00  01
    beat[  1]  01  ff  00  ff  00  01  00  ff  fd  00  01  01  00  00  00  00
    beat[  2]  00 

In [ ]:

# Step 1 — generate top.sv
# In DRAM mode the interface section of top.sv gains one set of ports per
# weight/bias parameter, e.g.:
#   input [fc1_WEIGHT_PRECISION_0-1:0] fc1_weight [...], fc1_weight_valid, output fc1_weight_ready
# instead of instantiating an fc1_weight_source BRAM ROM module.
mg_hw, _ = passes.emit_verilog_top_transform_pass(mg_hw)

# Step 2 — copy internal RTL component files (fixed_linear.sv, fixed_relu.sv, …)
mg_hw, _ = passes.emit_internal_rtl_transform_pass(mg_hw)

print("Emitted top.sv and internal RTL components.")
print("Output directory: ~/.mase/top/hardware/rtl/")


Emitted top.sv and internal RTL components.
Output directory: ~/.mase/top/hardware/rtl/


In [ ]:

# Step 3 — emit_bram pass (DRAM mode behaviour)
# In DRAM mode this pass does NOT generate ROM .sv / .dat files.
# It DOES delete any stale *_source.sv and *_rom.dat files that may have been
# generated by a previous BRAM run sitting in the same rtl/ directory.
# If verilator found those stale files it would compile them alongside the new
# DRAM top.sv and produce "illegal port connection" errors.
mg_hw, _ = passes.emit_bram_transform_pass(mg_hw)

print("BRAM pass complete (DRAM mode — no ROM files generated).")


BRAM pass complete (DRAM mode — no ROM files generated).


In [ ]:

# Step 4 — emit cocotb testbench
# In DRAM mode _emit_cocotb_tb creates a StreamDriver for every DRAM port
# (e.g. fc1_weight, fc1_bias, fc2_weight, fc2_bias, fc3_weight, fc3_bias).
# load_drivers() preloads each driver with the quantised 8-bit fixed-point
# parameter blocks, broken into PARALLELISM_DIM_0 × PARALLELISM_DIM_1 beats —
# exactly the format fixed_linear expects on its weight/bias handshake ports.
mg_hw, _ = passes.emit_cocotb_transform_pass(mg_hw)

print("Testbench emitted.")
print("DRAM StreamDrivers will be created for:")
for k in sorted(storage_map):
    node_name, param_name = k.split(".")
    print(f"  port: {node_name}_{param_name}  (valid / ready)")


Testbench emitted.
DRAM StreamDrivers will be created for:
  port: fc1_bias  (valid / ready)
  port: fc1_weight  (valid / ready)
  port: fc2_bias  (valid / ready)
  port: fc2_weight  (valid / ready)
  port: fc3_bias  (valid / ready)
  port: fc3_weight  (valid / ready)


In [ ]:

# Step 5 — simulate
# Verilator compiles all .sv files in ~/.mase/top/hardware/rtl/ and the
# mase_components library, then runs the cocotb test.
# The testbench will:
#   1. Drive input activation data via data_in_0 StreamDriver
#   2. Drive each weight/bias parameter via its dedicated DRAM StreamDriver
#   3. Check data_out_0 against the reference output from inference_model
from chop.actions import simulate

#simulate(skip_build=False, skip_test=False, waves=True)


## 6 — BRAM Mode: On-Chip Weight Storage

The DRAM section above streamed weights from **off-chip** memory via top-level handshake ports.  In the default **BRAM** mode each weight/bias tensor is stored in an **on-chip Block RAM** that is initialised at synthesis time from a `.dat` hex file and exposes the same valid/ready streaming interface internally — so `fixed_linear` sees the same protocol either way.

| Aspect | DRAM | BRAM |
|---|---|---|
| `top.sv` interface | Exposes `{node}_{param}` + `_valid/_ready` top-level ports | Only `data_in_0` / `data_out_0` visible externally |
| Weight source | Off-chip, one beat per inference cycle | On-chip ROM module, cycling autonomously after reset |
| `emit_bram` output | Deletes any stale ROM files | Generates `*_source.sv` + `*_rom.dat` per parameter |
| Cocotb testbench | `StreamDriver` per weight/bias port | No weight drivers — BRAM sources are self-contained |

### How the generated `*_source.sv` works

Three modules are stacked in each generated file:

```
*_rom      — synchronous single-port RAM; $readmemh("<param>_rom.dat", ram)
             at elaboration time.  One read port, 2-cycle latency.

*          — thin DATA_WIDTH / ADDR_RANGE wrapper around *_rom.

*_source   — cycling ROM source with valid/ready handshake:
               • counter advances on every beat where data_out_ready=1.
               • Wraps at OUT_DEPTH − 1 so the ROM cycles indefinitely.
               • data_out_valid is suppressed for the first 2 ready-beats
                 after reset to absorb the ROM read pipeline latency.
```

The `.dat` file encodes the weight tensor as hex rows, one row per `PARALLELISM_DIM_0 × PARALLELISM_DIM_1` block, values packed **little-endian within each row** (element [0] in the most-significant position of the hex string).

In [ ]:
# ── BRAM mode: build hardware graph with on-chip weight storage ───────────────
# We build a fresh MaseGraph from inference_model (plain nn.Linear weights),
# quantize it independently to 8-bit fixed-point, patch precision metadata, and
# add hardware metadata with the default BRAM storage.
# Note: quantize_transform_pass creates *new* LinearInteger module objects; it
# does not modify the original inference_model nn.Linear layers in-place.

mg_bram = MaseGraph(model=inference_model.cpu())
dummy_in_bram = {"x": torch.randn(1, 2)}

mg_bram, _ = passes.init_metadata_analysis_pass(mg_bram)
mg_bram, _ = passes.add_common_metadata_analysis_pass(
    mg_bram,
    {"dummy_in": dummy_in_bram, "add_value": False, "force_device_meta": False},
)

# Quantize to 8-bit fixed-point so the graph nodes have LinearInteger modules
# (which carry the .config dict that emit_bram uses to choose round vs floor).
hw_quant_config_bram = {
    "by": "type",
    "default": {"config": {"name": None}},
    "linear": {
        "config": {
            "name": "integer",
            "data_in_width": 8,
            "data_in_frac_width": 3,
            "weight_width": 8,
            "weight_frac_width": 3,
            "bias_width": 8,
            "bias_frac_width": 3,
        }
    },
}
mg_bram, _ = passes.quantize_transform_pass(mg_bram, pass_args=hw_quant_config_bram)

# Patch common metadata precision fields to reflect 8-bit fixed-point.
# (add_common_metadata sets type="float" / precision=[32]; we override here.)
for node in mg_bram.fx_graph.nodes:
    for arg_info in node.meta["mase"]["common"]["args"].values():
        if isinstance(arg_info, dict):
            arg_info["type"] = "fixed"
            arg_info["precision"] = [8, 3]
    for res_info in node.meta["mase"]["common"]["results"].values():
        if isinstance(res_info, dict):
            res_info["type"] = "fixed"
            res_info["precision"] = [8, 3]

# Omitting "interface.storage" (or passing "BRAM") keeps the default BRAM path.
mg_bram, _ = passes.add_hardware_metadata_analysis_pass(mg_bram)

# ── Verify BRAM metadata ──────────────────────────────────────────────────────
bram_storage_map = {}
for node in mg_bram.fx_graph.nodes:
    mase = node.meta.get("mase")
    if mase is None or mase.parameters["hardware"].get("is_implicit", False):
        continue
    iface = mase.parameters["hardware"].get("interface", {})
    for arg, info in mase.parameters["common"].get("args", {}).items():
        if "data_in" in arg or not isinstance(info, dict):
            continue
        bram_storage_map[f"{node.name}.{arg}"] = (
            iface.get(arg, {}).get("storage", "<missing>")
        )

assert bram_storage_map, "No parameter storage entries found."
bad = {k: v for k, v in bram_storage_map.items() if v != "BRAM"}
assert not bad, f"Expected all BRAM, got: {bad}"

print("BRAM hardware metadata verified — all parameters tagged BRAM:")
for k in sorted(bram_storage_map):
    print(f"  {k}: {bram_storage_map[k]}")

BRAM hardware metadata verified — all parameters tagged BRAM:
  fc1.bias: BRAM
  fc1.weight: BRAM
  fc2.bias: BRAM
  fc2.weight: BRAM
  fc3.bias: BRAM
  fc3.weight: BRAM


In [ ]:
# ── BRAM emit pipeline ────────────────────────────────────────────────────────
# Step 1: top.sv — weight/bias signals are driven by *_source modules
#         instantiated internally.  No top-level weight streaming ports.
mg_bram, _ = passes.emit_verilog_top_transform_pass(mg_bram)

# Step 2: Copy shared RTL component files (fixed_linear.sv, fixed_relu.sv, …)
mg_bram, _ = passes.emit_internal_rtl_transform_pass(mg_bram)

# Step 3: For each weight/bias parameter, write two files:
#
#   <node>_<param>_source.sv  — three stacked modules:
#       *_rom        : single-port synchronous RAM initialised from the .dat file
#       *            : thin wrapper used by *_source
#       *_source     : cycles the ROM and exposes a valid/ready streaming port;
#                      data_out_valid is held low for 2 cycles after reset
#                      to absorb the ROM read-pipeline latency, then stays high.
#
#   <node>_<param>_rom.dat    — $readmemh hex init file; one row per
#                               PARALLELISM_DIM_0 × PARALLELISM_DIM_1 slice,
#                               8-bit values packed little-endian within each row,
#                               rows ordered to match the streaming traversal order.
mg_bram, _ = passes.emit_bram_transform_pass(mg_bram)

print("BRAM emit complete.")
print("Output: ~/.mase/top/hardware/rtl/")

BRAM emit complete.
Output: ~/.mase/top/hardware/rtl/


In [ ]:
# ── Inspect generated ROM .dat files ─────────────────────────────────────────
# Each row is one parallelism block of weights encoded as hex.
# Row width = PRECISION_0 × PARALLELISM_DIM_0 × PARALLELISM_DIM_1 bits
# (= hex_chars × 4 bits).  The depth equals ceil(tensor_elements / parallelism).

from pathlib import Path

rtl_dir = Path.home() / ".mase" / "top" / "hardware" / "rtl"
sv_sources = sorted(rtl_dir.glob("*_source.sv"))
dat_files   = sorted(rtl_dir.glob("*_rom.dat"))

print(f"Source modules : {len(sv_sources)}")
for f in sv_sources:
    print(f"  {f.name}  ({f.stat().st_size:,} B)")

print(f"\nROM data files : {len(dat_files)}")
for f in dat_files:
    rows = f.read_text().strip().split("\n")
    hex_chars = len(rows[0]) if rows else 0
    print(f"  {f.name}  —  {len(rows)} rows × {hex_chars * 4} bits")

print("\nFirst 4 rows of each .dat file (hex, little-endian within row):")
for f in dat_files:
    rows = f.read_text().strip().split("\n")
    print(f"\n  {f.name}:")
    for i, row in enumerate(rows[:4]):
        print(f"    [{i:3d}]  0x{row}")

Source modules : 6
  fc1_bias_source.sv  (3,083 B)
  fc1_weight_source.sv  (3,153 B)
  fc2_bias_source.sv  (3,083 B)
  fc2_weight_source.sv  (3,157 B)
  fc3_bias_source.sv  (3,081 B)
  fc3_weight_source.sv  (3,155 B)

ROM data files : 6
  fc1_bias_rom.dat  —  16 rows × 32 bits
  fc1_weight_rom.dat  —  16 rows × 64 bits
  fc2_bias_rom.dat  —  16 rows × 32 bits
  fc2_weight_rom.dat  —  256 rows × 128 bits
  fc3_bias_rom.dat  —  1 rows × 32 bits
  fc3_weight_rom.dat  —  16 rows × 128 bits

First 4 rows of each .dat file (hex, little-endian within row):

  fc1_bias_rom.dat:
    [  0]  0x01faff02
    [  1]  0xfc00fd05
    [  2]  0xfffffc02
    [  3]  0x0602fd03

  fc1_weight_rom.dat:
    [  0]  0x0600fdfb00060800
    [  1]  0xfafa0005fc04fc04
    [  2]  0xfd050500000000f9
    [  3]  0xfd00fa00f9040403

  fc2_bias_rom.dat:
    [  0]  0x02ff0102
    [  1]  0x02000002
    [  2]  0x01020202
    [  3]  0x02010301

  fc2_weight_rom.dat:
    [  0]  0x00fefc0000f9ffffff00000102000001
    [  1]  0x0

In [ ]:
# ── BRAM cocotb testbench + simulation ────────────────────────────────────────
# In BRAM mode the testbench has no DRAM StreamDrivers.  Weights cycle out of
# the on-chip BRAM source modules automatically after reset is de-asserted, so
# the testbench only needs to drive data_in_0 and check data_out_0.
mg_bram, _ = passes.emit_cocotb_transform_pass(mg_bram)
print("BRAM testbench emitted.")

from chop.actions import simulate
simulate(skip_build=False, skip_test=False, waves=True)

BRAM testbench emitted.
INFO: Running command perl /usr/local/bin/verilator -cc --exe -Mdir /home/ism/ADL/mase/docs/source/modules/documentation/tutorials/sim_build -DCOCOTB_SIM=1 --top-module top --vpi --public-flat-rw --prefix Vtop -o top -LDFLAGS '-Wl,-rpath,/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/cocotb/libs -L/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/cocotb/libs -lcocotbvpi_verilator' --trace -Wno-fatal -Wno-lint -Wno-style --trace-fst --trace-structs --trace-depth 3 --build-jobs 1 --unroll-count 1024 -I/home/ism/.mase/top/hardware/rtl -I/home/ism/ADL/mase/src/mase_components/hls/rtl -I/home/ism/ADL/mase/src/mase_components/systolic_arrays/rtl -I/home/ism/ADL/mase/src/mase_components/activation_layers/rtl -I/home/ism/ADL/mase/src/mase_components/memory/rtl -I/home/ism/ADL/mase/src/mase_components/normalization_layers/rtl -I/home/ism/ADL/mase/src/mase_components/common/rtl -I/home/ism/ADL/mase/src/mase_components/language_models/rtl -I/home/ism/ADL/mase/sr

%Error: /home/ism/.mase/top/hardware/rtl/top.sv:489:27: Slices of arrays in assignments have different unpacked dimensions, 2 versus 4
  489 | assign fc1_data_in_0    = data_in_0;
      |                           ^~~~~~~~~
%Error: Exiting due to 1 error(s)


SystemExit: Process 'perl' terminated with error 1